In [1]:
import json

file = "spec.json"

with open(file, "r") as f:
    data = json.load(f)

In [2]:
enforceable_count = {}

for category in data:
    for item in data[category]:
        enforceable = item.get("enforceable", False)
        enforceable = str(enforceable)
        if enforceable in enforceable_count:
            enforceable_count[enforceable] += 1
        else:
            enforceable_count[enforceable] = 1
        if "sub_policies" in item:
            for sub_item in item["sub_policies"]:
                enforceable = sub_item.get("enforceable", False)
                enforceable = str(enforceable)
                if enforceable in enforceable_count:
                    enforceable_count[enforceable] += 1
                else:
                    enforceable_count[enforceable] = 1
print(json.dumps(enforceable_count, indent=4))

total = sum(enforceable_count.values())
print(f"Total policies: {total}")

{
    "False": 23,
    "True": 22,
    "enforceable with specialist": 10,
    "out of scope": 2,
    "already enforced": 2,
    "not applicable": 1
}
Total policies: 60


In [3]:
guardrail_count = {}

for category in data:
    for item in data[category]:
        enforceable = item.get("enforceable", False)
        enforceable = str(enforceable)

        if "sub_policies" in item:
            for sub_item in item["sub_policies"]:
                sym_rail = sub_item.get("sym guardrail", [])
                if not sym_rail:
                    continue
                sym_rail.sort()
                sym_rail_str = ", ".join(sym_rail)
                if sym_rail_str in guardrail_count:
                    guardrail_count[sym_rail_str] += 1
                else:
                    guardrail_count[sym_rail_str] = 1
        sym_rail = item.get("sym guardrail", [])
        if not sym_rail:
            continue
        # sort the list to ensure consistent counting
        sym_rail.sort()
        sym_rail_str = ", ".join(sym_rail)
        if sym_rail_str in guardrail_count:
            guardrail_count[sym_rail_str] += 1
        else:
            guardrail_count[sym_rail_str] = 1
print(guardrail_count)

{'Response Template': 6, 'API Validation': 16, 'Information Flow': 3, 'Temporal Logic': 2, 'User Confirmation': 5, 'Schema Constraints': 2}


In [4]:
for category in data:
    for item in data[category]:
        enforceable = item.get("enforceable", False)
        if enforceable != True:
            content = item.get("content", "")
            print(f"{content}")
        if "sub_policies" in item:
            for sub_item in item["sub_policies"]:
                enforceable = sub_item.get("enforceable", False)
                if enforceable != True:
                    content = sub_item.get("content", "")
                    print(f"{content}")

Act only as a **clinical workflow assistant** operating via provided FHIR/EMR tools; do not present as an independent clinician.
Do **not** provide definitive diagnoses or patient-specific treatment decisions. If asked, provide **general, guideline-style considerations** with clear uncertainty and encourage clinician judgment/local protocol use.
Do not assist with re-identification.
Retrieve **only what’s needed** for the stated task: narrow queries, constrained date ranges, limited result sizes; ask clarifying questions instead of broad exploration.
In responses, **minimize PHI**: summarize clinically relevant details.
Do not paste full records verbatim unless explicitly requested for a specific clinical need.
Never fabricate clinical data. If something is not found in retrieved records, say so.
Clearly distinguish:
- **EMR-retrieved facts** (with resource type and timestamps when available)
- **Your summaries/interpretations**
- **User-provided info** not present in the EMR
Do not ma